In [1]:
import sys
import numpy as np
import sounddevice as sd
import soundfile as sf
import pyqtgraph as pg
from PySide6.QtWidgets import (QApplication, QMainWindow, QVBoxLayout, QHBoxLayout, 
                               QWidget, QComboBox, QPushButton, QLabel, QFileDialog, QDoubleSpinBox, QMessageBox)

class AudioRecorderEditorApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Grabador y Editor de Ganancia Rode")
        self.resize(900, 750)

        # Variables de estado
        self.samplerate = 44100
        self.stream = None
        self.recorded_chunks = []
        self.recording_data = None
        self.loaded_data = None
        self.loaded_fs = None

        self.init_ui()
        self.populate_devices()

    def init_ui(self):
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QVBoxLayout(central_widget)

        # --- SECCIÓN 1: GRABACIÓN ---
        main_layout.addWidget(QLabel("<b>1. GRABACIÓN</b>"))
        
        self.device_combo = QComboBox()
        main_layout.addWidget(QLabel("Selecciona el micrófono:"))
        main_layout.addWidget(self.device_combo)

        record_layout = QHBoxLayout()
        self.btn_record = QPushButton("🔴 Iniciar Grabación")
        self.btn_record.setStyleSheet("background-color: #ffcccc; color: black;")
        self.btn_record.clicked.connect(self.start_recording)
        
        self.btn_stop = QPushButton("⏹ Detener Grabación")
        self.btn_stop.setEnabled(False)
        self.btn_stop.clicked.connect(self.stop_recording)
        
        self.btn_save_raw = QPushButton("💾 Guardar Grabación Original")
        self.btn_save_raw.setEnabled(False)
        self.btn_save_raw.clicked.connect(self.save_raw_audio)

        record_layout.addWidget(self.btn_record)
        record_layout.addWidget(self.btn_stop)
        record_layout.addWidget(self.btn_save_raw)
        main_layout.addLayout(record_layout)

        main_layout.addWidget(QLabel("<hr>"))

        # --- SECCIÓN 2: EDICIÓN Y VISUALIZACIÓN ---
        main_layout.addWidget(QLabel("<b>2. EDICIÓN Y GANANCIA</b>"))
        
        self.btn_load = QPushButton("📂 Abrir Archivo de Audio (.wav)")
        self.btn_load.clicked.connect(self.load_audio)
        main_layout.addWidget(self.btn_load)

        # Gráfica de la forma de onda
        self.plot_widget = pg.PlotWidget(title="Forma de Onda del Audio Cargado")
        self.plot_widget.setYRange(-1, 1)
        self.plot_widget.showGrid(x=True, y=True, alpha=0.3)
        self.waveform_curve = self.plot_widget.plot(pen='c')
        main_layout.addWidget(self.plot_widget)

        # Controles de Ganancia
        gain_layout = QHBoxLayout()
        gain_layout.addWidget(QLabel("Multiplicador de Ganancia (Ej. 2.0 = Doble de volumen):"))
        self.spin_gain = QDoubleSpinBox()
        self.spin_gain.setRange(0.0, 10.0) 
        self.spin_gain.setValue(1.0)
        self.spin_gain.setSingleStep(0.1)
        self.spin_gain.valueChanged.connect(self.update_plot_with_gain)
        gain_layout.addWidget(self.spin_gain)
        main_layout.addLayout(gain_layout)

        # --- NUEVO: Controles de Reproducción ---
        playback_layout = QHBoxLayout()
        self.btn_play = QPushButton("▶️ Escuchar con Ganancia")
        self.btn_play.setEnabled(False)
        self.btn_play.setStyleSheet("background-color: #cce5ff; color: black;")
        self.btn_play.clicked.connect(self.play_audio)
        
        self.btn_stop_play = QPushButton("⏹ Detener Reproducción")
        self.btn_stop_play.setEnabled(False)
        self.btn_stop_play.clicked.connect(self.stop_playback)

        playback_layout.addWidget(self.btn_play)
        playback_layout.addWidget(self.btn_stop_play)
        main_layout.addLayout(playback_layout)

        # Botón para guardar final
        self.btn_save_edited = QPushButton("💾 Guardar Audio con Ganancia Editada")
        self.btn_save_edited.setEnabled(False)
        self.btn_save_edited.setStyleSheet("background-color: #ccffcc; color: black;")
        self.btn_save_edited.clicked.connect(self.save_edited_audio)
        main_layout.addWidget(self.btn_save_edited)

    def populate_devices(self):
        for i, dev in enumerate(sd.query_devices()):
            if dev['max_input_channels'] > 0:
                self.device_combo.addItem(f"{dev['name']} ({dev['max_input_channels']} ch)", i)

    # --- FUNCIONES DE GRABACIÓN ---
    def audio_callback(self, indata, frames, time, status):
        if status:
            print(status, file=sys.stderr)
        self.recorded_chunks.append(indata.copy())

    def start_recording(self):
        dev_index = self.device_combo.currentData()
        dev_info = sd.query_devices(dev_index)
        channels = dev_info['max_input_channels']

        self.recorded_chunks = []
        self.stream = sd.InputStream(
            device=dev_index, channels=channels, 
            samplerate=self.samplerate, callback=self.audio_callback
        )
        self.stream.start()

        self.btn_record.setEnabled(False)
        self.btn_record.setText("Grabando...")
        self.btn_stop.setEnabled(True)
        self.btn_save_raw.setEnabled(False)

    def stop_recording(self):
        if self.stream:
            self.stream.stop()
            self.stream.close()
        
        if self.recorded_chunks:
            self.recording_data = np.vstack(self.recorded_chunks)
        
        self.btn_record.setEnabled(True)
        self.btn_record.setText("🔴 Iniciar Grabación")
        self.btn_stop.setEnabled(False)
        self.btn_save_raw.setEnabled(True)

    def save_raw_audio(self):
        if self.recording_data is None:
            return
        filepath, _ = QFileDialog.getSaveFileName(self, "Guardar Audio Original", "", "Archivos WAV (*.wav)")
        if filepath:
            sf.write(filepath, self.recording_data, self.samplerate)
            QMessageBox.information(self, "Éxito", "Audio original guardado correctamente.")

    # --- FUNCIONES DE EDICIÓN Y REPRODUCCIÓN ---
    def load_audio(self):
        filepath, _ = QFileDialog.getOpenFileName(self, "Abrir Archivo de Audio", "", "Archivos WAV (*.wav)")
        if filepath:
            self.loaded_data, self.loaded_fs = sf.read(filepath)
            
            if len(self.loaded_data.shape) > 1:
                plot_data = self.loaded_data[:, 0]
            else:
                plot_data = self.loaded_data

            self.waveform_curve.setData(plot_data)
            self.spin_gain.setValue(1.0) 
            
            # Habilitar botones de reproducción y guardado
            self.btn_play.setEnabled(True)
            self.btn_save_edited.setEnabled(True)

    def update_plot_with_gain(self):
        if self.loaded_data is None:
            return
        
        gain = self.spin_gain.value()
        
        if len(self.loaded_data.shape) > 1:
            plot_data = self.loaded_data[:, 0] * gain
        else:
            plot_data = self.loaded_data * gain
            
        self.waveform_curve.setData(plot_data)

    def play_audio(self):
        """Reproduce el audio aplicando la ganancia seleccionada."""
        if self.loaded_data is None:
            return
            
        gain = self.spin_gain.value()
        audio_to_play = self.loaded_data * gain 
        
        # Clip para proteger tus parlantes de volúmenes destructivos
        audio_to_play = np.clip(audio_to_play, -1.0, 1.0)
        
        # sounddevice reproduce en segundo plano automáticamente
        sd.play(audio_to_play, self.loaded_fs)
        self.btn_stop_play.setEnabled(True)

    def stop_playback(self):
        """Detiene la reproducción en curso."""
        sd.stop()
        self.btn_stop_play.setEnabled(False)

    def save_edited_audio(self):
        if self.loaded_data is None:
            return
            
        gain = self.spin_gain.value()
        edited_data = self.loaded_data * gain 
        edited_data = np.clip(edited_data, -1.0, 1.0)

        filepath, _ = QFileDialog.getSaveFileName(self, "Guardar Audio Editado", "", "Archivos WAV (*.wav)")
        if filepath:
            sf.write(filepath, edited_data, self.loaded_fs)
            QMessageBox.information(self, "Éxito", f"Audio guardado con una ganancia de {gain}x.")

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = AudioRecorderEditorApp()
    window.show()
    sys.exit(app.exec())

SystemExit: 0

c:\Users\Lenovo-PC\miniconda3\envs\PEMSA\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import os
os.path.join(os.path.dirname(os.path.dirname(__file__)), ".env")

NameError: name '__file__' is not defined